In [1]:
!curl -O https://www.amazontrust.com/repository/AmazonRootCA1.pem
!curl -O https://www.amazontrust.com/repository/AmazonRootCA2.pem
!curl -O https://www.amazontrust.com/repository/AmazonRootCA3.pem
!curl -O https://www.amazontrust.com/repository/AmazonRootCA4.pem
!curl -O https://certs.secureserver.net/repository/sf-class2-root.crt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  1188  100  1188    0     0   5416      0 --:--:-- --:--:-- --:--:--  5500
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  1883  100  1883    0     0  18451      0 --:--:-- --:--:-- --:--:-- 19214
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   656  100   656    0     0  10536      0 --:--:-- --:--:-- --:--:-- 11118
  % Total    % Received % Xferd  Average Speed   

In [2]:
# combine key files
!type AmazonRootCA1.pem AmazonRootCA2.pem AmazonRootCA3.pem AmazonRootCA4.pem sf-class2-root.crt > keyspaces-bundle.pem


AmazonRootCA1.pem



AmazonRootCA2.pem



AmazonRootCA3.pem



AmazonRootCA4.pem



sf-class2-root.crt




In [1]:
!pip install cassandra-sigv4

In [2]:
from cassandra.cluster import Cluster
from ssl import SSLContext, PROTOCOL_TLSv1_2 , CERT_REQUIRED
import boto3
from cassandra_sigv4.auth import SigV4AuthProvider

ssl_context = SSLContext(PROTOCOL_TLSv1_2)
ssl_context.load_verify_locations('access_keys/keyspaces-bundle.pem')
ssl_context.verify_mode = CERT_REQUIRED

C:\Users\tziga\AppData\Local\Temp\ipykernel_2380\1746359294.py:6: DeprecationWarning: ssl.PROTOCOL_TLSv1_2 is deprecated
  ssl_context = SSLContext(PROTOCOL_TLSv1_2)


In [3]:
import pandas as pd
access_key = pd.read_csv('access_keys/de300-keyspaces_accessKeys.csv')

In [ ]:
# use this if you want to use Boto to set the session parameters.
boto_session = boto3.Session(aws_access_key_id=access_key['Access key ID'].values[0],
                             aws_secret_access_key=access_key['Secret access key'].values[0],
                            #  aws_session_token="AQoDYXdzEJr...<remainder of token>",
                             region_name="us-east-1")
auth_provider = SigV4AuthProvider(boto_session)

cluster = Cluster(['cassandra.us-east-1.amazonaws.com'], 
                  ssl_context=ssl_context, 
                  auth_provider=auth_provider,
                  port=9142)
session = cluster.connect()
r = session.execute('select * from system_schema.keyspaces')
print(r.current_rows)

NoHostAvailable: ('Unable to connect to any servers', {'3.238.167.251:9142': AuthenticationFailed('Failed to authenticate to 3.238.167.251:9142: Error from server: code=0100 [Bad credentials] message="Authentication failure: AccessKeyDisabled"')})

In [5]:
# establishing connection to Keyspace
session = cluster.connect()

In [6]:
# Insert any CQL queries between .connect() and .shutdown()

# For example, show all keyspaces created
r = session.execute('''
    SELECT * FROM system_schema.keyspaces;
    ''')
print(r.current_rows)

[Row(keyspace_name='system_schema', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='system_schema_mcs', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='system', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='system_multiregion_info', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='de300_acharya', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='de300_barnett', durable_writes=True, replication=OrderedMa

In [11]:
q = '''
SELECT * FROM de300_chan.test_table;
'''

r = session.execute(q)

In [12]:
r.current_rows

[Row(country='US', user_id=1002, gender='F'),
 Row(country='GB', user_id=1001, gender='M')]

In [13]:
from cassandra import ConsistencyLevel
session.default_consistency_level = ConsistencyLevel.LOCAL_QUORUM

q = '''
INSERT INTO de300_chan.test_table (country, gender, user_id) VALUES ('US', 'F', 1002);
'''

r = session.execute(q)

C:\Users\tziga\AppData\Local\Temp\ipykernel_34952\829693375.py:2: DeprecationWarning: Setting the consistency level at the session level will be removed in 4.0. Consider using execution profiles and setting the desired consistency level to the EXEC_PROFILE_DEFAULT profile.
  session.default_consistency_level = ConsistencyLevel.LOCAL_QUORUM


In [14]:
# For example, create a keyspace for HW2
r = session.execute('''
    CREATE KEYSPACE IF NOT EXISTS de300_demo 
    WITH replication = {'class': 'SingleRegionStrategy'};
    ''')
print(r.current_rows)

[]


In [15]:
q = '''
SELECT * FROM de300_chan.test_table;
'''

r = session.execute(q)

NoHostAvailable: ('Unable to complete the operation against any hosts', {<Host: 3.238.167.236:9142 us-east-1>: ConnectionException('Host has been marked down or removed'), <Host: 3.238.167.251:9142 us-east-1>: ConnectionException('Host has been marked down or removed'), <Host: 3.234.248.199:9142 us-east-1>: ConnectionException('Host has been marked down or removed'), <Host: 3.234.248.207:9142 us-east-1>: ConnectionException('Host has been marked down or removed'), <Host: 3.234.248.208:9142 us-east-1>: ConnectionException('Host has been marked down or removed'), <Host: 3.234.248.236:9142 us-east-1>: ConnectionException('Host has been marked down or removed'), <Host: 3.234.248.255:9142 us-east-1>: ConnectionException('Host has been marked down or removed'), <Host: 3.234.248.195:9142 us-east-1>: ConnectionException('Host has been marked down or removed'), <Host: 3.234.248.220:9142 us-east-1>: ConnectionException('Host has been marked down or removed')})

In [12]:
session.shutdown()

# In Class Exercise (AWS Keyspace)

In [7]:
session = cluster.connect('de300_demo')

In [ ]:
# use age as the partition key and icd9_code as the clustering key
q = '''
CREATE TABLE IF NOT EXISTS de300_chan.procedures_by_patient_age (
    age int,
    icd9_code text,
    short_title text,
    PRIMARY KEY (age, icd9_code)
) WITH CLUSTERING ORDER BY (icd9_code ASC);
'''

session.execute(q)

SyntaxError: invalid syntax (2370004982.py, line 14)

In [ ]:
session.shutdown()